# OpenEnv Email Triage - Final Colab T4 Notebook

This notebook is prepared for **Google Colab Free Tier (T4 GPU)** and the repo:
- https://github.com/Rhushya/OpenEnv

Key rule:
- Keep shell commands (`!python ...`) and Python code (`print(...)`) in separate cells.

In [2]:
!nvidia-smi

Sat Apr 25 09:11:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/Rhushya/OpenEnv.git
%cd OpenEnv
!pip install -U pip
!pip install "torch>=2.3" "transformers>=4.46" "trl>=0.11.0" "accelerate>=0.34" datasets huggingface_hub bitsandbytes fastmcp

Cloning into 'OpenEnv'...
remote: Enumerating objects: 10499, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 10499 (delta 302), reused 387 (delta 263), pack-reused 10060 (from 3)
Receiving objects: 100% (10499/10499), 68.69 MiB | 30.29 MiB/s, done.
Resolving deltas: 100% (6249/6249), done.
/content/OpenEnv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 87.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 46.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 71.6 MB/s  0:00

## Smoke test (must pass)

In [4]:
!PYTHONPATH=src:envs python envs/email_triage_env/train_grpo.py --smoke

[SMOKE] Minimal run — just verifying pipeline works
[GPU] Tesla T4: 15.6 GB total, 15.6 GB free
[GPU] Precision: bf16
[INFO] Unsloth not found — using standard HuggingFace loading
[INFO] Install unsloth for 2x faster training on Colab T4
config.json: 100% 661/661 [00:00<00:00, 2.52MB/s]
tokenizer_config.json: 1.29kB [00:00, 4.00MB/s]
vocab.json: 2.78MB [00:00, 53.8MB/s]
merges.txt: 1.67MB [00:00, 101MB/s]
tokenizer.json: 7.03MB [00:00, 125MB/s]
[TOKENIZER] Loaded tokenizer for Qwen/Qwen2-0.5B
[GPU] After model load: 15.6 GB free
Traceback (most recent call last):
  File "/content/OpenEnv/envs/email_triage_env/train_grpo.py", line 329, in <module>
    main()
  File "/content/OpenEnv/envs/email_triage_env/train_grpo.py", line 246, in main
    config = GRPOConfig(
             ^^^^^^^^^^^
TypeError: GRPOConfig.__init__() got an unexpected keyword argument 'max_prompt_length'


In [5]:
print("Smoke test complete. If this passed, run full training.")

Smoke test complete. If this passed, run full training.


## Full T4 training run

In [6]:
!PYTHONPATH=src:envs python envs/email_triage_env/train_grpo.py --model Qwen/Qwen2-0.5B --max-steps 50 --dataset-size 64 --output-dir oversight-arena-grpo-t4

Traceback (most recent call last):
  File "/content/OpenEnv/envs/email_triage_env/train_grpo.py", line 329, in <module>
    main()
  File "/content/OpenEnv/envs/email_triage_env/train_grpo.py", line 161, in main
    from trl import GRPOConfig, GRPOTrainer
  File "/usr/local/lib/python3.12/dist-packages/trl/__init__.py", line 19, in <module>
    from . import _compat
  File "/usr/local/lib/python3.12/dist-packages/trl/_compat.py", line 27, in <module>
object address  : 0x7cf21278d3c0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [ ]:
print("\nTraining complete. Checkpoint saved to oversight-arena-grpo-t4/")

## Push model to Hugging Face Hub

In [ ]:
!huggingface-cli login

In [ ]:
!PYTHONPATH=src:envs python envs/email_triage_env/train_grpo.py --model Qwen/Qwen2-0.5B --max-steps 50 --dataset-size 64 --output-dir oversight-arena-grpo-t4 --push-to-hub --hub-repo Rhushya/oversight-arena-grpo-t4

## Troubleshooting

- `ModuleNotFoundError: fastmcp` -> rerun install cell.
- `ModuleNotFoundError: core` -> pull latest repo and rerun.
- CUDA OOM -> use `--max-steps 30 --dataset-size 32`.
- If installs were changed, restart runtime before rerun.